# Controlling LLM Output

LangChain provides following features to adjust the output relevance:
- **Temperature:**
- **max_tokens:**
- **top_p:**
- **top_k:**

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

from dotenv import  load_dotenv
load_dotenv()

True

In [2]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", # Mandatory - selects model
)

llm.invoke("What's rainy season?")

AIMessage(content="The **rainy season**, also known as the **wet season** or **monsoon season** (in regions where monsoons are the primary driver), is a period of the year characterized by significantly higher average precipitation than other times of the year.\n\nHere's a breakdown of what that means:\n\n1.  **Increased Rainfall:** The most obvious characteristic is a sustained period of heavy and frequent rainfall. This can range from daily downpours to several days of continuous rain, depending on the region.\n2.  **Geographical Occurrence:** It primarily occurs in tropical and subtropical regions of the world. Temperate regions generally have more evenly distributed rainfall throughout the year, or different seasonal patterns.\n3.  **Causes:** The rainy season is typically caused by:\n    *   **Monsoons:** Large-scale seasonal wind shifts that bring moist air from oceans over landmasses (e.g., South Asia, Southeast Asia, parts of Africa).\n    *   **Intertropical Convergence Zone (

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=2.0,
    # temperature is the Randomness parameter 
    # - Decides how variated the output should be
    # - range 0.0 to 2.0
)

llm.invoke("What's rainy season?")

AIMessage(content="The **rainy season**, also known as the **wet season** or **monsoon season** in some regions, is an annual period of increased and sustained precipitation that occurs in tropical and subtropical climates around the globe.\n\nHere's a breakdown of what it is:\n\n1.  **High Precipitation:** It is marked by significantly higher average rainfall compared to the rest of the year. The rain is often heavy, prolonged, and frequent, sometimes accompanied by thunderstorms.\n2.  **Duration:** It typically lasts for several weeks to several months, depending on the specific location and its climatic patterns.\n3.  **Humidity:** High humidity is common during the rainy season due to increased evaporation and consistent rainfall.\n4.  **Location:** It's predominantly found in tropical and subtropical regions. This includes areas affected by monsoon winds (like South Asia, Southeast Asia, parts of Africa), as well as regions near the equator where the Intertropical Convergence Zone

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    max_output_tokens=500, 
    # max_output_tokens is the Upper Limit for tokens consumption.
    # - Model TRIES to stay within given token range.
    # - NOTE: this is max_tokens in OpenAI
)

llm.invoke("What's rainy season?")

AIMessage(content='The **rainy season**, also known as the **wet season** or **monsoon season', additional_kwargs={}, response_metadata={'finish_reason': 'MAX_TOKENS', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019fb292-e804-74a0-ad60-5554525c1397-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 496, 'total_tokens': 503, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 477}})

# How Temperature Works?

Before understanding the temperature param, let's understand how LLM works:
- LLM is a "next word prediction machine" which selects most appropriate words for given input.
- For this it evaluates thousands of probabilities using it's **vocab**.
- Vocab is all the words (around 1,000,000+) available to LLM on which it is trained on.
- Words with highest probabilities are selected.

This all happens under the following process:
1. **Next-Word Prediction**: The model evaluates thousands of possible next words in its vocabulary.
2. **Logits (Raw Scores)**: Before calculating probabilities, the model gives each word a raw score called a **logit** (e.g., *rain* = 10, *water* = 6, *banana* = 1).
3. **Softmax Function**: Softmax converts these raw scores into final probabilities (percentages between 0.0 and 1.0) that sum up to 1.0 (100%).

---

## Where Temperature is used here?

When we provide a Temperature ($T$) parameter, every logit is divided with it.

$$\text{Adjusted Score} = \frac{\text{Logit}}{\text{Temperature}}$$

### 1. Low Temperature (e.g., `0.0` to `0.3`)
- When logits are divided with a lower temperature value, the gap between top score and low score increases.
- Thus, the top score words can be chosen with focus, consistency, and factual strictness.
- **Best for**: Coding, Math, Fact Extraction, Q&A.

### 2. High Temperature (e.g., `1.0` to `2.0`)
- The gap between top score and low score decreases, and inappropriate words sometimes have same scores.
- Sometimes, the low score words get selected. The output becomes creative, diverse, and unexpected.
- **Best for**: Storytelling, Brainstorming, Poetry, Creative Writing.

## Quick Cheat Sheet

| Temperature | Behavior | Output Style |
| :--- | :--- | :--- |
| **0.0 - 0.2** | Deterministic & Precise | Same response every time, highly factual |
| **0.7 - 1.0** | Balanced & Natural | Default standard responses |
| **1.5 - 2.0** | Random & Creative | Highly varied, unexpected phrasing |

---

## Softmax Function

In [ ]:
import numpy as np

# Raw scores (logits) for candidate words: ["rain", "water", "banana"]
logits = np.array([10.0, 6.0, 1.0])

def calculate_softmax(logits, temperature=1.0):
    # 1. Divide raw scores by temperature
    scaled_logits = logits / temperature
    # 2. Exponentiate scaled scores
    exp_values = np.exp(scaled_logits)
    # 3. Normalize to get probabilities (sum to 1.0)
    probabilities = exp_values / np.sum(exp_values)
    return probabilities

print("Raw Scores (Logits):", logits)
print("Low Temp (0.2)  Probabilities:", np.round(calculate_softmax(logits, temperature=0.2), 4))
print("Normal Temp (1.0) Probabilities:", np.round(calculate_softmax(logits, temperature=1.0), 4))
print("High Temp (2.0) Probabilities:", np.round(calculate_softmax(logits, temperature=2.0), 4))


Raw Scores (Logits): [10.  6.  1.]
Low Temp (0.2)  Probabilities: [1. 0. 0.]
Normal Temp (1.0) Probabilities: [9.819e-01 1.800e-02 1.000e-04]
High Temp (2.0) Probabilities: [0.8723 0.118  0.0097]


That's how the probabilities get affected based on the provided temperature.

---

# Top_P (Nucleus Sampling)

**Top_P** (also known as Nucleus Sampling) selects only top words whose sum of probability reaches the `top_p` value.

### How Top_P Works:
1. All possible next words are sorted from **highest to lowest probability**.
2. The probabilities are summed starting from the top word.
3. As soon as the sum reaches `top_p` (e.g., `0.9`), all remaining words below are **discarded**.
4. The model samples the next word **only from the selected pool**.

### Example: `top_p = 0.9`

Suppose an LLM evaluates the following words for the next position:
<table>
  <thead>
    <tr>
      <th>Words</th>
      <th>Probability</th>
      <th>Cumulative Sum</th>
      <th>Selected</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><b>India</b></td>
      <td>0.30</td>
      <td>0.30</td>
      <td rowspan="5" style="vertical-align: middle; text-align: center;"><b>Selected</b><br/>(Cumulative Sum = 0.90)</td>
    </tr>
    <tr>
      <td><b>capital</b></td>
      <td>0.20</td>
      <td>0.50</td>
    </tr>
    <tr>
      <td><b>is</b></td>
      <td>0.20</td>
      <td>0.70</td>
    </tr>
    <tr>
      <td><b>The</b></td>
      <td>0.10</td>
      <td>0.80</td>
    </tr>
    <tr>
      <td><b>of</b></td>
      <td>0.10</td>
      <td>0.90</td>
    </tr>
    <tr>
      <td>apple</td>
      <td>0.05</td>
      <td>0.95</td>
      <td rowspan="2" style="vertical-align: middle; text-align: center;"><b>Discarded</b><br/>(Exceeds 0.90 threshold)</td>
    </tr>
    <tr>
      <td>blue</td>
      <td>0.05</td>
      <td>1.00</td>
    </tr>
  </tbody>
</table>
